<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial - GarmentIQ Classification

GarmentIQ classification identifies which category a garment image belongs to, such as a
short sleeve top, a vest dress, or a skirt. It is the first step of the measurement
pipeline, because every later stage needs to know the garment type before it can choose
the right landmarks and measurement instructions.

This tutorial shows how to load the pretrained tinyViT classifier, predict the category of
a single image, and evaluate the model across a whole dataset.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Classify a single image](#single)
3. [Evaluate on a dataset](#dataset)

<a name="prerequisites"></a>
## Prerequisites

Install the package, then download the test dataset and the pretrained weights. On Colab
you can keep this section collapsed.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import torch

import garmentiq as giq
from garmentiq.classification.model_definition import tinyViT
from garmentiq.classification.utils import CachedDataset

# GarmentIQ never grabs an accelerator on its own: every model loader and every
# inference function takes a `device` argument that defaults to "cpu". Pass it
# explicitly to use a GPU ("cuda") or Apple Silicon ("mps").
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

In [ ]:
# @title Download the dataset and the pretrained model

!mkdir -p models

!curl -sL -o garmentiq-classification-set-nordstrom-and-myntra.zip \
  https://www.kaggle.com/api/v1/datasets/download/lygitdata/garmentiq-classification-set-nordstrom-and-myntra

!wget -q -O ./models/tiny_vit.pt \
    https://huggingface.co/lygitdata/garmentiq/resolve/main/tiny_vit.pt

print("Downloads finished.")

<a name="single"></a>
## Classify a single image

Every GarmentIQ model follows the same two-step shape: load the model once, then call the
inference function with it. Classification is no exception.

In [ ]:
# Split the archive into a train and test set, then cache the test images
DATA = giq.classification.train_test_split(
    output_dir="data",
    metadata_csv="metadata.csv",
    label_column="garment",
    train_zip_dir="garmentiq-classification-set-nordstrom-and-myntra.zip",
    test_size=0.15,
    verbose=True,
)

In [ ]:
# Step 1: load the model
classifier = giq.classification.load_model(
    model_path="./models/tiny_vit.pt",
    model_class=tinyViT,
    model_args={"num_classes": 9, "img_size": (120, 184), "patch_size": 6},
    device=device,
)

In [ ]:
# Step 2: predict. `classes` maps the model's output index onto a readable label.
img_to_test = DATA["test_metadata"]["filename"][88]

pred_label, pred_prob = giq.classification.predict(
    model=classifier,
    image_path=f"data/test/images/{img_to_test}",
    classes=DATA["test_metadata"]["garment"].unique().tolist(),
    resize_dim=(120, 184),
    normalize_mean=[0.8047, 0.7808, 0.7769],
    normalize_std=[0.2957, 0.3077, 0.3081],
    device=device,
)

print("Image:", img_to_test)
print("Predicted label:", pred_label)
print("Predicted probabilities:", pred_prob)

<a name="dataset"></a>
## Evaluate on a dataset

`test_pytorch_nn` runs the model over a whole dataset and reports accuracy. It loads the
weights itself, so it takes the model path rather than a loaded model.

In [ ]:
test_images, test_labels, _ = giq.classification.load_data(
    df=DATA["test_metadata"],
    img_dir=DATA["test_images"],
    label_column="garment",
    resize_dim=(120, 184),
    normalize_mean=[0.8047, 0.7808, 0.7769],
    normalize_std=[0.2957, 0.3077, 0.3081],
)

In [ ]:
giq.classification.test_pytorch_nn(
    model_path="./models/tiny_vit.pt",
    model_class=tinyViT,
    model_args={"num_classes": 9, "img_size": (120, 184), "patch_size": 6},
    dataset_class=CachedDataset,
    dataset_args={
        "raw_labels": DATA["test_metadata"]["garment"],
        "cached_images": test_images,
        "cached_labels": test_labels,
    },
    # This function takes its device inside `param`, alongside the batch size
    param={"batch_size": 64, "device": device},
)

Training and fine-tuning your own classifier are covered in the
advanced usage notebooks.